# 08 Program Generation Agent

**The central AI agent: intake form → complete individual program**

## Flow
```
User fills intake form
        ↓
Visual BF% assessment (Gemini Vision - user photos vs reference PDF)
        ↓
Goal validation (validate_goal - cut/bulk/maintain based on BF%)
        ↓
Run all calculators (TDEE, macros, 1RM, optimal volume)
        ↓
RAG retrieval - course principles + case studies as structural templates
        ↓
GPT-4o-mini synthesizes complete program in Bulgarian
        ↓
Structured output: training program + nutrition plan + targets
```

## Case study role
Case studies are used as **structural templates only** - they show how a program
looks and how decisions are made. All actual numbers come from:
- Calculators (TDEE, macros, 1RM, volume)
- Course principles via RAG (exercise selection, frequency, periodization)
- User's own data (intake form)

In [ ]:
import json, os, sys, base64
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import faiss
from openai import OpenAI
from dotenv import load_dotenv
from tenacity import retry, wait_exponential, stop_after_attempt
import google.generativeai as genai
load_dotenv(override=True)

NOTEBOOK_DIR = Path().resolve()
BACKEND_DIR = NOTEBOOK_DIR.parent
VS_DIR = BACKEND_DIR / 'data' / 'vectorstore'
PROCESSED_DIR = BACKEND_DIR / 'data' / 'processed'
VISUAL_REF_DIR = PROCESSED_DIR / 'visual_reference'

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
GEMINI = genai.GenerativeModel(os.getenv('GEMINI_VISION_MODEL'))

EMBED_MODEL = os.getenv('EMBEDDING_MODEL')
LLM_MODEL = os.getenv('PRIMARY_MODEL')
EMBED_DIM = 3072
RETRIEVAL_K = 20
FINAL_K = 7

# Add calculators to path
sys.path.insert(0, str(BACKEND_DIR / 'tools'))
import calculators as Calc

print('Setup complete')

Setup complete


## 1. Load FAISS + Metadata

In [3]:
index = faiss.read_index(str(VS_DIR / 'henselmans_openai.index'))
metadata = json.loads((VS_DIR / 'henselmans_openai_metadata.json').read_text(encoding='utf-8'))
assert index.ntotal == len(metadata)
print(f'FAISS: {index.ntotal:,} vectors | Metadata: {len(metadata):,} records')

FAISS: 6,077 vectors | Metadata: 6,077 records


## 2. Core RAG Functions

In [4]:
def embed_query(text: str) -> np.ndarray:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=[text])
    emb = np.array(r.data[0].embedding, dtype='float32').reshape(1, -1)
    faiss.normalize_L2(emb)
    return emb

def mmr_diversity(candidates: list, top_n: int = FINAL_K, lambda_: float = 0.7) -> list:
    """Maximal Marginal Relevance - balance relevance vs source diversity."""
    if len(candidates) <= top_n:
        return candidates
    selected, remaining = [candidates[0]], list(candidates[1:])
    while len(selected) < top_n and remaining:
        best_score, best_idx = -999, 0
        for j, cand in enumerate(remaining):
            same_src = sum(1 for s in selected if s['source'] == cand['source'])
            mmr = lambda_ * cand['score'] - (1 - lambda_) * same_src * 0.2
            if mmr > best_score:
                best_score, best_idx = mmr, j
        selected.append(remaining.pop(best_idx))
    return selected

def retrieve(query: str, k: int = RETRIEVAL_K,
             filter_meta: dict = None, include_case_studies: bool = True) -> list:
    q_emb = embed_query(query)
    scores, ids = index.search(q_emb, k * 2)
    results = []
    for s, idx in zip(scores[0], ids[0]):
        if idx == -1: continue
        m = metadata[idx]
        if filter_meta and not all(m.get(fk) == fv for fk, fv in filter_meta.items()):
            continue
        if not include_case_studies and m.get('is_case_study'):
            continue
        results.append({**m, 'score': float(s)})
    return sorted(results, key=lambda x: x['score'], reverse=True)[:k]

def dual_retrieve_mmr(q1: str, q2: str, filter_meta: dict = None,
                      include_case_studies: bool = True) -> list:
    seen = {}
    for r in (retrieve(q1, filter_meta=filter_meta, include_case_studies=include_case_studies) +
              retrieve(q2, filter_meta=filter_meta, include_case_studies=include_case_studies)):
        cid = r['chunk_id']
        if cid not in seen or r['score'] > seen[cid]['score']:
            seen[cid] = r
    merged = sorted(seen.values(), key=lambda x: x['score'], reverse=True)
    return mmr_diversity(merged[:30], top_n=FINAL_K)

print('RAG functions defined')

RAG functions defined


## 3. Visual BF% Assessment (Gemini Vision)

In [5]:
def assess_bf_visually(user_photo_paths: list[str]) -> dict:
    """
    Compare user photos against the visual reference guide using Gemini Vision.
    Returns estimated BF% range and confidence.
    Falls back to BMI estimate if no photos provided.
    """
    manifest_path = VISUAL_REF_DIR / 'manifest.json'
    if not manifest_path.exists():
        return {'method': 'no_reference', 'bf_pct': None, 'confidence': 'low'}

    manifest   = json.loads(manifest_path.read_text(encoding='utf-8'))
    ref_pages  = list(manifest.get('pages', {}).values())

    if not user_photo_paths:
        return {'method': 'no_photos', 'bf_pct': None, 'confidence': 'none'}

    # Build Gemini Vision prompt with reference images + user photos
    parts = []

    # Add reference guide pages (first 4 pages cover the key BF% ranges)
    parts.append('REFERENCE GUIDE - Body Fat Percentage Visual Examples:')
    for page_path in ref_pages[:4]:
        if Path(page_path).exists():
            img_data = Path(page_path).read_bytes()
            parts.append({'mime_type': 'image/png', 'data': base64.b64encode(img_data).decode()})

    # Add user photos
    parts.append('USER PHOTOS (front, back, side):')
    for photo_path in user_photo_paths:
        if Path(photo_path).exists():
            img_data = Path(photo_path).read_bytes()
            mime_type = 'image/jpeg' if photo_path.lower().endswith('.jpg') else 'image/png'
            parts.append({'mime_type': mime_type, 'data': base64.b64encode(img_data).decode()})

    parts.append("""
Compare the user's physique to the reference guide examples.
Estimate their body fat percentage range.
Consider: muscle definition visibility, fat distribution, vascularity.
Be realistic and conservative in your estimate.

Respond in JSON:
{
  "bf_pct_low": <lower bound>,
  "bf_pct_high": <upper bound>,
  "bf_pct_estimate": <midpoint>,
  "confidence": "high|medium|low",
  "reasoning": "brief explanation"
}
Reply ONLY with valid JSON.""")

    try:
        response = GEMINI.generate_content(parts)
        result = json.loads(response.text)
        result['method'] = 'visual_gemini'
        return result
    except Exception as e:
        return {'method': 'visual_error', 'bf_pct': None, 'error': str(e), 'confidence': 'none'}


def get_bf_estimate(user_profile: dict) -> float:
    """
    Get BF% estimate - visual if photos available, BMI fallback otherwise.
    Updates user_profile in place with bf_pct and bf_method.
    """
    photos = user_profile.get('photo_paths', [])

    if photos:
        result = assess_bf_visually(photos)
        bf_pct = result.get('bf_pct_estimate')
        if bf_pct:
            user_profile['body_fat_pct'] = bf_pct
            user_profile['bf_method'] = f'visual ({result.get("confidence","?")} confidence)'
            return bf_pct

    # BMI fallback
    result = Calc.estimate_bf_bmi(
        age = user_profile['age'],
        sex = user_profile['sex'],
        height_cm = user_profile['height_cm'],
        bodyweight_kg = user_profile['bodyweight_kg'],
    )
    user_profile['body_fat_pct'] = result.bf_percentage
    user_profile['bf_method'] = 'BMI estimate (no photos)'
    return result.bf_percentage


print('Visual BF% assessment defined')
print('  With photos: Gemini Vision vs reference guide')
print('  Without photos: BMI formula (fallback)')

Visual BF% assessment defined
  With photos: Gemini Vision vs reference guide
  Without photos: BMI formula (fallback)


## 4. Intake Form Definition

In [6]:
# This mirrors the Henselmans Coaching Client Intake Form exactly
# All fields the AI needs to generate a complete program

INTAKE_FORM_FIELDS = {
    # Personal data
    'name': str,
    'age': int,
    'sex': str,      # 'male' | 'female'
    'height_cm': float,
    'bodyweight_kg': float,
    'photo_paths': list,     # optional - for visual BF% assessment

    # Goal
    'goal': str,      # 'bulk' | 'cut' | 'maintain' | 'aggressive_cut'
    'goal_details': str,      # free text - what they want to achieve

    # Activity
    'activity_level': str,      # 'sedentary'|'light'|'moderate'|'active'|'very_active'
    'activity_details': str,      # job type, daily movement description

    # Training history
    'training_status': int,      # 1=novice, 2=intermediate, 3=advanced
    'training_years': float,    # years of consistent training
    'training_days_per_week': int,      # available training days
    'available_equipment': str,      # 'full_gym' | 'home_basic' | 'home_full' | 'barbell_only'
    'session_duration_min': int,      # available time per session in minutes

    # Current lifts (for 1RM estimation)
    'lifts': dict,     # {lift: {weight_kg, reps}}

    # Preferences & constraints
    'priority_muscles': list,     # muscles to prioritize
    'injuries': str,      # injury history
    'exercise_preferences': str,      # likes/dislikes
    'dietary_restrictions': str,      # vegan, allergies, etc.
}

print('Intake form fields defined')
print(f'Total fields: {len(INTAKE_FORM_FIELDS)}')

Intake form fields defined
Total fields: 20


## 5. Program Generation Agent

In [7]:
PROGRAM_SYSTEM_PROMPT = """Ти си персонален фитнес треньор и нутриционист, работещ по методологията на Menno Henselmans.

ЗАДАЧА: Изготви пълна индивидуална програма за клиента на базата на:
1. Данните от intake формата
2. Резултатите от калкулаторите (TDEE, макроси, 1RM, оптимален обем)
3. Принципите от курса на Henselmans (предоставени в контекста)
4. Case studies като структурен шаблон - виж как изглежда добре изготвена програма, 
   НО не копирай числата от тях. Числата идват САМО от калкулаторите и данните на клиента.

ПРАВИЛА:
- Отговаряй САМО на БЪЛГАРСКИ
- Използвай КИЛОГРАМИ и метри
- Бъди конкретен - давай точни упражнения, серии, повторения, тежести
- Обясни ЗАЩО взимаш всяко решение (кратко)
- Програмата трябва да е директно приложима - клиентът да може да я следва от утре

ФОРМАТ НА ИЗХОДА:
Структурирай програмата в следните секции:
1. Профил и анализ
2. Хранителен план (калории, макроси, разпределение)
3. Тренировъчна програма (split, упражнения, серии, повторения, тежести)
4. Прогресия (как да увеличава натоварването)
5. Цели и проследяване (какво да мери и кога)
"""

@retry(wait=wait_exponential(min=1, max=20), stop=stop_after_attempt(3))
def generate_program(user_profile: dict, calc_results: dict, context_chunks: list) -> str:
    """Generate complete individual program in Bulgarian."""

    # Build calculator results summary
    e = calc_results['energy']
    v = calc_results['volume']
    gv = calc_results['goal_validation']

    calc_summary = f"""
РЕЗУЛТАТИ ОТ КАЛКУЛАТОРИТЕ:

Телесен състав:
- Тегло: {user_profile['bodyweight_kg']} кг
- BF%: {user_profile.get('body_fat_pct', 'N/A')}% ({user_profile.get('bf_method', '')})
- LBM: {e.lbm_kg} кг

Енергия:
- BMR: {e.bmr_kcal} ккал
- TDEE: {e.tdee_kcal} ккал
- Целева калории ({e.goal}): {e.target_kcal} ккал
- Протеин: {e.protein_g} г/ден
- Мазнини: {e.fat_g} г/ден
- Въглехидрати: {e.carbs_g} г/ден

Цел: {'ПРЕПОРЪКА ЗА ПРОМЯНА: ' + gv.override_reason if gv.override else f'✅ {e.goal}'}

Оптимален обем (серии/седмица):
{chr(10).join(f'- {m}: {s} серии' for m, s in v.muscle_groups.items())}

1RM резултати:
{chr(10).join(f'- {lift}: {orm.estimated_1rm} кг' for lift, orm in calc_results["lifts"].items())}
"""

    # Build context from RAG
    context = '\n\n'.join(
        f'[{"CASE STUDY - структурен шаблон" if c["is_case_study"] else "Курс материал"}: {c["source"]}]\n{c["text"]}'
        for c in context_chunks
    )

    # Build user profile summary
    profile_summary = f"""
КЛИЕНТСКИ ПРОФИЛ:
- Име: {user_profile.get('name', 'Клиент')}
- Възраст: {user_profile['age']} г., {user_profile['sex']}
- Ниво: {'Начинаещ' if user_profile['training_status']==1 else 'Intermediate' if user_profile['training_status']==2 else 'Напреднал'}
- Тренировъчни дни: {user_profile['training_days_per_week']}/седмица
- Оборудване: {user_profile.get('available_equipment', 'full_gym')}
- Продължителност: {user_profile.get('session_duration_min', 60)} мин/тренировка
- Приоритети: {', '.join(user_profile.get('priority_muscles', [])) or 'няма специфични'}
- Контузии: {user_profile.get('injuries', 'няма')}
- Хранителни ограничения: {user_profile.get('dietary_restrictions', 'няма')}
- Цел: {user_profile.get('goal_details', '')}
"""

    messages = [
        {'role': 'system', 'content': PROGRAM_SYSTEM_PROMPT},
        {'role': 'user', 'content': f"""{profile_summary}

{calc_summary}

КОНТЕКСТ ОТ КУРСА НА HENSELMANS:
{context}

Изготви пълна индивидуална програма за този клиент."""}
    ]

    response = openai_client.chat.completions.create(
        model = LLM_MODEL,
        messages = messages,
        temperature = 0.4,
        max_tokens = 3000,
    )
    return response.choices[0].message.content


print('Program generation agent defined')

Program generation agent defined


## 6. Full Program Generation Pipeline

In [8]:
def generate_full_program(user_profile: dict) -> dict:
    """
    Complete pipeline: intake form data → full program.

    Steps:
    1. BF% assessment (visual or BMI)
    2. Goal validation
    3. Run all calculators
    4. RAG retrieval (principles + case study templates)
    5. Generate program
    """
    print(f'Generating program for {user_profile.get("name", "client")}...')

    # Step 1: BF% assessment
    print('[1/5] BF% assessment...')
    bf_pct = get_bf_estimate(user_profile)
    print(f'BF%: {bf_pct}% ({user_profile.get("bf_method")})')

    # Step 2 & 3: Goal validation + all calculators
    print('  [2/5] Running calculators...')
    calc_results = Calc.run_all_calculators(user_profile)
    gv = calc_results['goal_validation']
    if gv.override:
        print(f'Goal override: {user_profile["goal"]} → {gv.recommended_goal}')
        print(f'Reason: {gv.override_reason[:80]}...')
    else:
        print(f'Goal confirmed: {gv.recommended_goal}')

    e = calc_results['energy']
    print(f'TDEE: {e.tdee_kcal} kcal | Target: {e.target_kcal} kcal')
    print(f'Macros: P{e.protein_g}g F{e.fat_g}g C{e.carbs_g}g')

    # Step 4: RAG retrieval
    print('[3/5] Retrieving course principles...')
    level_map = {1: 'novice', 2: 'intermediate', 3: 'advanced'}
    level = level_map[user_profile['training_status']]
    goal = calc_results['goal_validation'].recommended_goal

    # Query 1: Training principles for level + goal
    q_training = f'{level} training program {goal} {user_profile["training_days_per_week"]} days per week volume frequency'
    # Query 2: Nutrition principles for goal
    q_nutrition = f'protein intake macros {goal} {level} cutting bulking energy balance'
    # Query 3: Case study template (structural reference)
    q_casestudy = f'{level} {"male" if user_profile["sex"]=="male" else "female"} {goal} program design example'

    # Retrieve principles (no case studies)
    principle_chunks = dual_retrieve_mmr(q_training, q_nutrition,
                                          include_case_studies=False)
    # Retrieve case study templates
    cs_chunks = mmr_diversity(
        sorted([c for c in retrieve(q_casestudy, k=10)
                if c.get('is_case_study')],
               key=lambda x: x['score'], reverse=True),
        top_n=2
    )
    context_chunks = principle_chunks + cs_chunks
    print(f'Principles: {len(principle_chunks)} chunks | Case study templates: {len(cs_chunks)}')

    # Step 5: Generate program
    print('[4/5] Generating program (GPT-4o-mini)...')
    program_text = generate_program(user_profile, calc_results, context_chunks)

    print('[5/5] Done!')

    return {
        'user_profile': user_profile,
        'calc_results': {
            'energy': vars(calc_results['energy']),
            'volume': {'muscle_groups': calc_results['volume'].muscle_groups,
                                  'notes': calc_results['volume'].notes},
            'goal_validation': {'recommended_goal': gv.recommended_goal,
                                  'override': gv.override,
                                  'override_reason': gv.override_reason,
                                  'bf_status': gv.bf_status},
            'lifts': {k: vars(v) for k, v in calc_results['lifts'].items()},
        },
        'context_sources': [c['source'] for c in context_chunks],
        'program': program_text,
    }


print('Full pipeline defined: generate_full_program(user_profile)')

Full pipeline defined: generate_full_program(user_profile)


## 7. Test - Generate Program for Example User

In [9]:
# Example user — intermediate male, cutting
test_profile = {
    'name': 'Тест Потребител',
    'age': 28,
    'sex': 'male',
    'height_cm': 180,
    'bodyweight_kg': 88,
    'body_fat_pct': 18,     # pre-assessed or from photos
    'photo_paths': [],     # no photos in test
    'goal': 'cut',
    'goal_details': 'Искам да се изчистя, да запазя мускулите си и да стигна около 12% BF',
    'activity_level': 'moderate',
    'activity_details': 'Офис работа, ходя пеш ~20 мин/ден',
    'training_status': 2,
    'training_years': 3.0,
    'training_days_per_week': 4,
    'available_equipment': 'full_gym',
    'session_duration_min': 75,
    'lifts': {
        'bench': {'weight': 100, 'reps': 5},
        'squat': {'weight': 120, 'reps': 5},
        'deadlift': {'weight': 150, 'reps': 3},
        'ohp': {'weight': 65, 'reps': 6},
    },
    'priority_muscles': ['back', 'shoulders'],
    'injuries': 'лека болка в дясното рамо при overhead движения',
    'exercise_preferences': 'Обичам compound движения, не харесвам leg press',
    'dietary_restrictions': 'няма',
}

result = generate_full_program(test_profile)

print()
print('=' * 65)
print('GENERATED PROGRAM')
print('=' * 65)
print(result['program'])

Generating program for Тест Потребител...
[1/5] BF% assessment...
BF%: 22.8% (BMI estimate (no photos))
  [2/5] Running calculators...
Goal confirmed: cut
TDEE: 2905 kcal | Target: 2405 kcal
Macros: P176g F70g C268g
[3/5] Retrieving course principles...
Principles: 7 chunks | Case study templates: 2
[4/5] Generating program (GPT-4o-mini)...
[5/5] Done!

GENERATED PROGRAM
### 1. Профил и анализ
**Име:** Тест Потребител  
**Възраст:** 28 г.  
**Пол:** Мъж  
**Ниво:** Intermediate  
**Тренировъчни дни:** 4/седмица  
**Оборудване:** Пълна зала  
**Продължителност:** 75 мин/тренировка  
**Приоритети:** Гръб, рамене  
**Контузии:** Лека болка в дясното рамо при overhead движения  
**Цел:** Изчистване на мазнини до около 12% BF, запазване на мускулна маса  

### 2. Хранителен план
**Целева калории (cut):** 2405 ккал  
**Макроси:**  
- **Протеин:** 176 г (704 ккал)  
- **Мазнини:** 70 г (630 ккал)  
- **Въглехидрати:** 268 г (1072 ккал)  

**Разпределение на макросите:**  
- **Закуска:** 30% о

In [10]:
print()
print('=' * 65)
print('CALCULATOR RESULTS SUMMARY')
print('=' * 65)
cr = result['calc_results']
e  = cr['energy']
print(f'BF Method: {result["user_profile"].get("bf_method")}')
print(f'Goal: {cr["goal_validation"]["recommended_goal"]} (override: {cr["goal_validation"]["override"]})')
print(f'TDEE: {e["tdee_kcal"]} kcal | Target: {e["target_kcal"]} kcal')
print(f'Macros: P{e["protein_g"]}g  F{e["fat_g"]}g  C{e["carbs_g"]}g')
print()
print('1RM estimates:')
for lift, data in cr['lifts'].items():
    print(f'  {lift:<12}: {data["estimated_1rm"]} кг')
print()
print('Weekly volume (sets):')
for muscle, sets in cr['volume']['muscle_groups'].items():
    print(f'  {muscle:<15}: {sets} серии/седмица')
print()
print(f'Context sources used ({len(result["context_sources"])}):')
for src in result['context_sources']:
    print(f'  - {src[:55]}')


CALCULATOR RESULTS SUMMARY
BF Method: BMI estimate (no photos)
Goal: cut (override: False)
TDEE: 2905 kcal | Target: 2405 kcal
Macros: P176g  F70g  C268g

1RM estimates:
  bench       : 116.7 кг
  squat       : 140.0 кг
  deadlift    : 165.0 кг
  ohp         : 78.0 кг

Weekly volume (sets):
  chest          : 12 серии/седмица
  back           : 18 серии/седмица
  shoulders      : 16 серии/седмица
  biceps         : 10 серии/седмица
  triceps        : 10 серии/седмица
  quads          : 14 серии/седмица
  hamstrings     : 10 серии/седмица
  glutes         : 12 серии/седмица
  calves         : 10 серии/седмица
  abs            : 8 серии/седмица
  rear_delts     : 10 серии/седмица

Context sources used (9):
  - Training case studies PTC 2022.pdf
  - Protein PTC 2022.pdf
  - Henselmans Energy intake calculator.xlsx
  - Energy PTC 2022.pdf
  - Training volume frequency intensity PTC 2022.pdf
  - Fasting and biorhythms PTC 2022.pdf
  - Protein PTC 2022.pdf
  - Case study average Joe program

## 8. Test - Goal Override (bulk at high BF%)

In [11]:
print('TEST: User wants to bulk but BF% is too high')
print('=' * 55)

override_profile = {
    'name': 'Override Test',
    'age': 25,
    'sex': 'male',
    'height_cm': 178,
    'bodyweight_kg': 90,
    'body_fat_pct': 22,     # too high for bulk
    'photo_paths': [],
    'goal': 'bulk', # user WANTS to bulk
    'goal_details': 'Искам да кача мускулна маса',
    'activity_level': 'moderate',
    'training_status': 2,
    'training_years': 2.0,
    'training_days_per_week': 4,
    'available_equipment': 'full_gym',
    'session_duration_min': 60,
    'lifts': {
        'bench': {'weight': 80, 'reps': 8},
        'squat': {'weight': 100, 'reps': 6},
        'deadlift': {'weight': 120, 'reps': 5},
    },
    'priority_muscles': [],
    'injuries': 'няма',
    'exercise_preferences': '',
    'dietary_restrictions': '',
}

# Just run calculators to show the goal override logic
gv_test = Calc.validate_goal(22, 'male', 'bulk')
print(f'User stated goal : bulk')
print(f'BF%: 22% (threshold: >15% for males)')
print(f'Override: {gv_test.override}')
print(f'Recommended goal: {gv_test.recommended_goal}')
print(f'BF status: {gv_test.bf_status}')
print(f'Reason: {gv_test.override_reason}')
print()
print('The agent will generate a CUT program and explain why,')
print('even though the user asked for bulk.')

TEST: User wants to bulk but BF% is too high
User stated goal : bulk
BF%: 22% (threshold: >15% for males)
Override: True
Recommended goal: cut
BF status: too_high
Reason: BF% (22%) is above the optimal range for males (>15.0%). Cutting first improves insulin sensitivity, nutrient partitioning, and long-term muscle gain potential.

The agent will generate a CUT program and explain why,
even though the user asked for bulk.


## 9. Summary

In [12]:
print('=' * 60)
print('  NOTEBOOK 08 - COMPLETE')
print('=' * 60)
print()
print('  Program generation pipeline:')
print('  1. Visual BF% (Gemini Vision) or BMI fallback')
print('  2. Goal validation (cut/bulk/maintain based on BF%)')
print('  3. All calculators (TDEE, macros, 1RM, volume)')
print('  4. RAG: principles (no case studies) + 2 case study templates')
print('  5. GPT-4o-mini generates complete Bulgarian program')
print()
print('  Case studies role: structural templates only')
print('  Numbers: ONLY from calculators + user data')

  NOTEBOOK 08 — COMPLETE

  Program generation pipeline:
  1. Visual BF% (Gemini Vision) or BMI fallback
  2. Goal validation (cut/bulk/maintain based on BF%)
  3. All calculators (TDEE, macros, 1RM, volume)
  4. RAG: principles (no case studies) + 2 case study templates
  5. GPT-4o-mini generates complete Bulgarian program

  Case studies role: structural templates only
  Numbers: ONLY from calculators + user data
